In [1]:
import pandas as pd
import numpy as np
import os
import shutil
import re
from pathlib import Path

In [ ]:
def summary_pivot_table(df, index_col, col, val):
    temp_df = df.copy()
    temp_df["Elapsed Minutes"] = pd.to_timedelta(temp_df['Elapsed Time'])
    temp_df["Elapsed Minutes"] = (temp_df['Elapsed Minutes'].dt.total_seconds() / 60).round(2)
    pivot_table_df = pd.pivot_table(temp_df, index=index_col, columns=col, values=val, aggfunc="mean", fill_value=0).stack().reset_index()
    if isinstance(index_col, list):
        group_cols = index_col + [col]
        new_cols = index_col + [col, "Average Elapsed Minutes"]
    else:
        group_cols = [index_col, col]
        new_cols = [index_col, col, "Average Elapsed Minutes"]
    pivot_table_df.columns = new_cols
    pivot_table_df["Average Elapsed Minutes"] = pivot_table_df["Average Elapsed Minutes"].round(2)
    temp_df["Total Failed Jobs"] = (temp_df["Status Code"] > 1).astype(int)
    temp_df["Total Successful Jobs"] = (temp_df["Status Code"] <= 1).astype(int)
    job_status_summary = temp_df.groupby(index_col + [col])[["Total Failed Jobs", "Total Successful Jobs"]].sum().reset_index()
    job_latest_success = (temp_df[temp_df["Status Code"] <= 1].groupby(group_cols)["End Time"].max().reset_index().rename(columns={"End Time": "Latest Successful End Time"}))
    pivot_table_df = pivot_table_df.merge(job_status_summary, on=index_col + [col], how="left")
    pivot_table_df = pivot_table_df.merge(job_latest_success, on=group_cols, how="left")
    return pivot_table_df

In [6]:
policies_list_df = pd.read_csv("data/email_dl.csv")

In [32]:
# source_folders = [Path(r"C:\Users\User\Desktop\Projects\netbackup_weekly_report\data\May_2026")]
source_folders = [Path(r"/Users/willis/Projects/netbackup_weekly_report/data/May_2026")]

all_dfs = []
all_failed_dfs = []

for source_folder in source_folders:
    report_csv_files = source_folder.rglob("daily_backup_report_*")
    failed_report_csv_files = source_folder.rglob("daily_baselined_failed_backup_report_*")
    for report_csv_file in report_csv_files:
        try:
            df = pd.read_csv(report_csv_file)
            df = df.drop(columns = 'Number of Child Jobs')
            if 'Failure Count' in df.columns and 'Backup Selection' in df.columns:
                df = df.drop(columns=['Failure Count', 'Backup Selection'], errors='ignore')
            all_dfs.append(df)
        except Exception as e:
            print(f"Error : {e}")
    for failed_report_csv_file in failed_report_csv_files:
        try:
            df = pd.read_csv(failed_report_csv_file)
            # df = df.drop(columns = 'Number of Child Jobs')
            if 'Backup Selection' in df.columns:
                df = df.drop(columns='Backup Selection', errors='ignore')
            all_failed_dfs.append(df)
        except Exception as e:
            print(f"Error : {e}")

# combine all csv into one dataframe
may_df = pd.concat(all_dfs, ignore_index=True)
may_df["Start Time"] = pd.to_datetime(may_df["Start Time"], errors="coerce")
may_df["End Time"] = pd.to_datetime(may_df["End Time"], errors="coerce")

# combine all csv into one dataframe
may_failed_df = pd.concat(all_failed_dfs, ignore_index=True)
may_failed_df["Start Time"] = pd.to_datetime(may_failed_df["Start Time"], errors="coerce")
may_failed_df["End Time"] = pd.to_datetime(may_failed_df["End Time"], errors="coerce")

may_df_copy = may_df.copy()
may_failed_df_copy = may_failed_df.copy()

may_df_copy["Start Date"] = may_df_copy["Start Time"].dt.date
may_failed_df_copy["Start Date"] = may_failed_df_copy["Start Time"].dt.date

In [48]:
may_df_copy = may_df_copy.sort_values(by=['Client', 'Policy', 'Backup Selection', 'Start Date'], ascending=True).reset_index(drop=True)
# may_df_copy.to_csv('test.csv', index=False)
may_df_copy["Successful Job"] = may_df_copy["Status Code"].isin([0,1])
result_df = (may_df_copy.groupby(["Client", "Policy", "Backup Selection", "Start Date"])["Successful Job"].any()).reset_index()
result_df

,Client,Policy,Backup Selection,Start Date,Successful Job
0,APP-SRV-01,APP-DAILY,C:\,2026-05-01,False
1,APP-SRV-01,APP-DAILY,C:\,2026-05-02,False
2,APP-SRV-01,APP-DAILY,C:\,2026-05-03,False
3,APP-SRV-01,APP-DAILY,C:\,2026-05-04,True
4,APP-SRV-01,APP-DAILY,C:\,2026-05-06,True
...,...,...,...,...,...
131,WEB-SRV-02,WEB-WEEKLY,E:\,2026-05-05,False
132,WEB-SRV-02,WEB-WEEKLY,E:\,2026-05-06,True
133,WEB-SRV-02,WEB-WEEKLY,E:\,2026-05-07,False
134,WEB-SRV-02,WEB-WEEKLY,E:\,2026-05-09,True


In [17]:
parent_jobs = set(may_failed_df["Job"].astype(str))

may_failed_filtered_df = may_failed_df[
    ~may_failed_df["Parent Job"].astype(str).isin(parent_jobs)
].copy()

may_failed_filtered_df["Start Day"] = may_failed_filtered_df["Start Time"].dt.date
may_failed_filtered_df = may_failed_filtered_df.sort_values(["Client", "Policy", "Start Time"])

may_failed_filtered_df = may_failed_filtered_df.drop_duplicates(subset=["Start Day", "Policy", "Client"], keep="first")

may_failed_filtered_df

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Failure Count,Start Day
10,8900000,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,58,2026-05-01 00:00:00,2026-05-01 01:53:00,1:53:00,4,2026-05-01
11,8900001,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,2,2026-05-02 00:00:00,2026-05-02 01:32:00,1:32:00,6,2026-05-02
12,8900002,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,150,2026-05-03 00:00:00,2026-05-03 02:38:00,2:38:00,2,2026-05-03
13,8900003,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,0,2026-05-04 00:00:00,2026-05-04 01:49:00,1:49:00,4,2026-05-04
14,8900004,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,1,2026-05-05 00:00:00,2026-05-05 01:00:00,1:00:00,8,2026-05-05
15,8900005,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-06 00:00:00,2026-05-06 00:51:00,0:51:00,2,2026-05-06
16,8900006,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-07 00:00:00,2026-05-07 00:11:00,0:11:00,3,2026-05-07
17,8900007,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-08 00:00:00,2026-05-08 01:38:00,1:38:00,10,2026-05-08
0,8868641,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-19 17:01:37,2026-05-19 17:26:46,25m9s,1,2026-05-19
2,8871834,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,150,2026-05-20 08:56:38,2026-05-20 20:31:16,11h34m38s,2,2026-05-20


In [ ]:
# may_failed_filtered_df.loc[may_failed_filtered_df['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

In [18]:
may_failed_filtered_df_copy = may_failed_filtered_df.copy()
may_failed_filtered_df_copy = may_failed_filtered_df_copy.sort_values(["Client", "Policy", "Start Time"])

may_failed_filtered_df_copy["New Incident"] = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy"])["Failure Count"]
      .diff()
      .ne(1)
)

may_failed_filtered_df_copy["Incident ID"] = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy"])["New Incident"]
      .cumsum()
)

may_failed_filtered_df_copy

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Failure Count,Start Day,New Incident,Incident ID
10,8900000,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,58,2026-05-01 00:00:00,2026-05-01 01:53:00,1:53:00,4,2026-05-01,True,1
11,8900001,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,2,2026-05-02 00:00:00,2026-05-02 01:32:00,1:32:00,6,2026-05-02,True,2
12,8900002,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,150,2026-05-03 00:00:00,2026-05-03 02:38:00,2:38:00,2,2026-05-03,True,3
13,8900003,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,0,2026-05-04 00:00:00,2026-05-04 01:49:00,1:49:00,4,2026-05-04,True,4
14,8900004,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,1,2026-05-05 00:00:00,2026-05-05 01:00:00,1:00:00,8,2026-05-05,True,5
15,8900005,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-06 00:00:00,2026-05-06 00:51:00,0:51:00,2,2026-05-06,True,6
16,8900006,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-07 00:00:00,2026-05-07 00:11:00,0:11:00,3,2026-05-07,False,6
17,8900007,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-08 00:00:00,2026-05-08 01:38:00,1:38:00,10,2026-05-08,True,7
0,8868641,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,10,2026-05-19 17:01:37,2026-05-19 17:26:46,25m9s,1,2026-05-19,True,8
2,8871834,BACKUP,Daily,-,APP-SRV-01,APP-DAILY,DONE,150,2026-05-20 08:56:38,2026-05-20 20:31:16,11h34m38s,2,2026-05-20,False,8


In [ ]:
# may_failed_filtered_df_copy.loc[may_failed_filtered_df_copy['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

In [19]:
incident_summary = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy", "Incident ID"])
      .agg(
          Start=("Start Time", "min"),
          End=("Start Time", "max"),
          MaxFailureCount=("Failure Count", "max")
      )
      .reset_index()
)

incident_summary

,Client,Policy,Incident ID,Start,End,MaxFailureCount
0,APP-SRV-01,APP-DAILY,1,2026-05-01 00:00:00,2026-05-01 00:00:00,4
1,APP-SRV-01,APP-DAILY,2,2026-05-02 00:00:00,2026-05-02 00:00:00,6
2,APP-SRV-01,APP-DAILY,3,2026-05-03 00:00:00,2026-05-03 00:00:00,2
3,APP-SRV-01,APP-DAILY,4,2026-05-04 00:00:00,2026-05-04 00:00:00,4
4,APP-SRV-01,APP-DAILY,5,2026-05-05 00:00:00,2026-05-05 00:00:00,8
5,APP-SRV-01,APP-DAILY,6,2026-05-06 00:00:00,2026-05-07 00:00:00,3
6,APP-SRV-01,APP-DAILY,7,2026-05-08 00:00:00,2026-05-08 00:00:00,10
7,APP-SRV-01,APP-DAILY,8,2026-05-19 17:01:37,2026-05-20 08:56:38,2
8,DB-SRV-01,DB-DAILY,1,2026-04-30 10:56:35,2026-04-30 10:56:35,9
9,DB-SRV-01,DB-DAILY,2,2026-05-01 00:00:00,2026-05-01 00:00:00,8


In [ ]:
# incident_summary.loc[incident_summary['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

In [ ]:
incident_summary.to_csv("failed_streak_may_2.csv", index=False)

In [ ]:
policy_df = {}
policy_summary_df = {}

output_path = Path("report_output/")
if not output_path.exists():
    output_path.mkdir()

for pattern in policies_list_df['Policy'].dropna().unique():
    filtered_df_all = may_df[
        may_df['Policy'].astype(str).str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )].copy().sort_values(by=['Client', 'Policy'], ascending=True).reset_index(drop=True)
    filtered_df = filtered_df_all.loc[filtered_df_all['Parent Job'] == '-']
    filtered_df = filtered_df.sort_values(by=['Client', 'Policy'], ascending=True)
    filtered_pivot_df = summary_pivot_table(filtered_df.copy(), index_col=["Client", "Policy"], col="State", val="Elapsed Minutes")
    policy_df[pattern] = filtered_df_all
    policy_summary_df[pattern] = filtered_pivot_df

In [31]:
# for name, df in policy_df.items():
#     df.to_csv(f"{output_path}/{name}_report.csv", index=False)

for name in policy_df.keys():
    destination_file = f"{output_path}/{name}_report.xlsx"
    if os.path.exists(destination_file):
            os.remove(destination_file)
    with pd.ExcelWriter(destination_file, engine="openpyxl") as writer:

        policy_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Report",
            index=False
        )

        policy_summary_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Summary",
            index=False
        )